# DailyMed Acquisition — US RX+OTC pill images + metadata

**Run this with Accelerator = None (CPU)** — it only downloads and parses
data, no GPU needed. Keeping it off the GPU means it doesn't touch your
GPU-hours quota while it runs.

**Run it via "Save Version" -> "Save & Run All (Commit)"**, not interactively.
Interactive edit-mode sessions idle-time-out if you step away (this is what
wiped an earlier in-progress download); a committed run executes as a batch
job on Kaggle's servers and isn't affected by your browser/laptop being
closed.

## What this produces

After a successful commit, this notebook's Output (under `/kaggle/working/`)
will contain:
- `dailymed_manifest.csv` — one row per resolved image: `path, label(NDC),
  side, domain, source, category(RX/OTC)`.
- `dailymed_metadata.json` — per-NDC name/imprint/color/shape/score, keyed by
  NDC (for the cascading filter, not just training).
- `dailymed_images/` — the actual resolved image files.

**Attach this output to the training notebook** via Add Input -> search ->
filter "Notebook" -> pick this notebook. Its output files will be readable
under `/kaggle/input/<this-notebook's-slug>/` once attached (exact path
printed by the training notebook's own dataset-discovery cell — same
`find_dataset_root`-style search used for every other dataset in this
project, since Kaggle's exact mount path has proven inconsistent across
input types in practice).

## Scope (edit this if you want more/less)

Default is deliberately conservative: 2 OTC parts (the priority gap) + 1 RX
part, not all 17 parts DailyMed actually has. Each part is several GB over
FTP plus tens of thousands of nested per-document zips to parse — widen
`SPL_PARTS_TO_FETCH` in Section 1 once you've confirmed this default scope
completes and the output looks right.


## 0. Setup

In [ ]:
import json
import re
import time
import zipfile
import urllib.request
import urllib.parse
from dataclasses import dataclass, field as _field
from io import BytesIO
from pathlib import Path

import requests

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(exist_ok=True)
SPL_IMAGE_DIR = WORK_DIR / "dailymed_images"
SPL_IMAGE_DIR.mkdir(exist_ok=True)
MANIFEST_PATH = WORK_DIR / "dailymed_manifest.csv"
METADATA_PATH = WORK_DIR / "dailymed_metadata.json"
STATE_PATH = WORK_DIR / "acquisition_state.json"

print("Setup OK. Working directory:", WORK_DIR)


## 1. Scope + discover current zip URLs

Fetches the real, live DailyMed resources page and regex-scans it for the
actual current `.zip` links — DailyMed's part count has grown over time (11
OTC + 6 RX parts confirmed live as of this notebook's last edit, vs. only
2+3 assumed in 2016-era references), so this discovers whatever exists now
rather than trusting a hardcoded list.


In [ ]:
SPL_RESOURCES_PAGE = "https://dailymed.nlm.nih.gov/dailymed/spl-resources-all-drug-labels.cfm"

# Conservative default — widen once this scope has been confirmed to work.
SPL_PARTS_TO_FETCH = ["human_otc_part1", "human_otc_part2", "human_rx_part1"]

# Safety cap per part so one huge part can't silently run for many hours
# unattended. None = no cap (process every document in the part).
MAX_DOCS_PER_PART = None

def discover_spl_zip_urls():
    resp = requests.get(SPL_RESOURCES_PAGE, timeout=30)
    resp.raise_for_status()
    hrefs = re.findall(
        r'href="([^"]*dm_spl_release_(?:human_rx|human_otc)_part\d+\.zip)"',
        resp.text,
    )
    urls = {}
    for href in hrefs:
        url = href if (href.startswith("http") or href.startswith("ftp")) else f"https://dailymed.nlm.nih.gov{href}"
        m = re.search(r"(human_rx_part\d+|human_otc_part\d+)\.zip", url)
        if m:
            urls[m.group(1)] = url
    return urls

spl_zip_urls = discover_spl_zip_urls()
print(f"Discovered {len(spl_zip_urls)} SPL zip URLs total:")
for part, url in sorted(spl_zip_urls.items()):
    marker = " <-- WILL FETCH" if part in SPL_PARTS_TO_FETCH else ""
    print(f"  {part}: {url}{marker}")

missing = [p for p in SPL_PARTS_TO_FETCH if p not in spl_zip_urls]
if missing:
    print(f"\nWARNING: requested parts not found on the page: {missing}")


## 2. SPL parser (inlined — no external file dependency)

Extracts NDC, name, imprint/color/shape/score-marks, and referenced image
filenames from each SPL XML. Only oral solid dosage forms (tablets/capsules)
are kept — liquids, injections, patches, etc. are filtered out via the NCI
Thesaurus dosage-form code whitelist below (ported from HHS's archived
`pillbox-data-process` repo, the actual source of this extraction approach).


In [ ]:
SPL_NS = {"v3": "urn:hl7-org:v3"}
ORAL_SOLID_DOSAGE_FORM_CODES = {
    "C25158", "C42895", "C42896", "C42917", "C42902", "C42904", "C42916",
    "C42928", "C42936", "C42954", "C42998", "C42893", "C42897", "C60997",
    "C42905", "C42997", "C42910", "C42927", "C42931", "C42930", "C61004",
    "C61005", "C42964", "C42963", "C42999", "C61006", "C42985", "C42992",
}

@dataclass
class SplPillRecord:
    ndc: str
    setid: str | None
    name: str | None
    rx_or_otc: str
    imprint: str | None = None
    color: str | None = None
    shape: str | None = None
    score_marks: str | None = None
    size_mm: str | None = None
    image_refs: list = _field(default_factory=list)

def _text(el):
    return el.text.strip() if el is not None and el.text else None

def _document_image_refs(root):
    # Confirmed against a live sample: real DailyMed documents don't use the
    # SPLIMAGE-characteristic pattern the archived pillbox-data-process
    # script assumed. Images are referenced via <observationMedia><value
    # mediaType="image/..."><reference value="foo.jpg"/></value></observationMedia>
    # blocks under separate package-label sections, not nested under the
    # product/characteristics at all. Typically package/box photos, not
    # always an isolated loose-pill shot.
    refs = []
    for om in root.iterfind(".//v3:observationMedia", SPL_NS):
        ref = om.find("./v3:value/v3:reference", SPL_NS)
        if ref is not None and ref.get("value"):
            refs.append(ref.get("value"))
    return refs

def parse_spl_bytes(xml_bytes, rx_or_otc):
    from lxml import etree
    root = etree.fromstring(xml_bytes)
    document_image_refs = _document_image_refs(root)
    setid_el = root.find(".//v3:setId", SPL_NS)
    setid = setid_el.get("root") if setid_el is not None else None
    records = []
    for product in root.iterfind(".//v3:manufacturedProduct", SPL_NS):
        form_code_el = product.find("./v3:formCode", SPL_NS)
        form_code = form_code_el.get("code") if form_code_el is not None else None
        if form_code not in ORAL_SOLID_DOSAGE_FORM_CODES:
            continue
        ndc_codes = sorted({
            c.get("code") for c in product.iterfind(".//v3:code", SPL_NS)
            if c.get("code") and "-" in c.get("code")
        })
        if not ndc_codes:
            continue
        name = _text(product.find(".//v3:name", SPL_NS))
        attrs = {"SPLCOLOR": [], "SPLIMPRINT": [], "SPLSHAPE": [], "SPLSCORE": [], "SPLSIZE": [], "SPLIMAGE": []}
        for characteristic in product.iterfind(".//v3:subjectOf/v3:characteristic", SPL_NS):
            code_el = characteristic.find("./v3:code", SPL_NS)
            if code_el is None:
                continue
            ctype = code_el.get("code")
            if ctype not in attrs:
                continue
            if ctype == "SPLIMPRINT":
                text = _text(characteristic.find("./v3:value", SPL_NS))
                if text:
                    attrs[ctype].append(text)
            elif ctype == "SPLIMAGE":
                ref = characteristic.find(".//v3:reference", SPL_NS)
                if ref is not None and ref.get("value"):
                    attrs[ctype].extend(ref.get("value").split())
            else:
                value_el = characteristic.find("./v3:value", SPL_NS)
                if value_el is not None:
                    v = value_el.get("displayName") or value_el.get("code") or value_el.get("value")
                    if v:
                        attrs[ctype].append(v)
        for ndc in ndc_codes:
            records.append(SplPillRecord(
                ndc=ndc, setid=setid, name=name, rx_or_otc=rx_or_otc,
                imprint=";".join(attrs["SPLIMPRINT"]) or None,
                color=",".join(attrs["SPLCOLOR"]) or None,
                shape=attrs["SPLSHAPE"][0] if attrs["SPLSHAPE"] else None,
                score_marks=attrs["SPLSCORE"][0] if attrs["SPLSCORE"] else None,
                size_mm=attrs["SPLSIZE"][0] if attrs["SPLSIZE"] else None,
                image_refs=list(dict.fromkeys(attrs["SPLIMAGE"] + document_image_refs)),
            ))
    return records

print("Parser defined.")


## 3. Download + process each part

Confirmed structure (via a live sample pull): each top-level bulk zip is
itself a **zip of per-document zips** — e.g. `otc/20090619_....zip` — and
each of those nested zips is flat (no subfolders): one XML plus its
referenced image(s) sitting as direct siblings (`bonine-01.jpg`,
`bonine-02.jpg`, `<uuid>.xml`).

Each part is downloaded, processed, and its results appended to the
manifest CSV **immediately** (not batched until the very end) and recorded
in a small state file — so if this part-by-part loop is interrupted
partway, already-completed parts are skipped on a re-run rather than
reprocessed from scratch. FTP downloads get a few retries since a transient
NIH FTP hiccup shouldn't cost hours of unattended progress.


In [ ]:
import csv as _csv

def load_state():
    return json.load(open(STATE_PATH)) if STATE_PATH.exists() else {}

def save_state(state):
    json.dump(state, open(STATE_PATH, "w"), indent=2)

def append_manifest_rows(rows):
    file_exists = MANIFEST_PATH.exists()
    with open(MANIFEST_PATH, "a", newline="") as f:
        w = _csv.DictWriter(f, fieldnames=["path", "label", "side", "domain", "source", "category"])
        if not file_exists:
            w.writeheader()
        w.writerows(rows)

def merge_metadata(records):
    existing = json.load(open(METADATA_PATH)) if METADATA_PATH.exists() else {}
    for rec in records:
        existing[rec.ndc] = {
            "name": rec.name, "imprint": rec.imprint, "color": rec.color,
            "shape": rec.shape, "score_marks": rec.score_marks,
            "size_mm": rec.size_mm, "rx_or_otc": rec.rx_or_otc,
        }
    json.dump(existing, open(METADATA_PATH, "w"), indent=0)

def download_to(url, dest_path, retries=3):
    last_err = None
    for attempt in range(retries):
        try:
            if url.startswith("ftp://"):
                urllib.request.urlretrieve(url, dest_path)
            else:
                resp = requests.get(url, stream=True, timeout=120)
                resp.raise_for_status()
                with open(dest_path, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
            return
        except Exception as e:
            last_err = e
            wait = 5 * (attempt + 1)
            print(f"  download attempt {attempt+1}/{retries} failed ({e}); retrying in {wait}s")
            time.sleep(wait)
    raise last_err

def process_part(part, url, rx_or_otc):
    print(f"\n=== {part} ===")
    tmp_zip_path = WORK_DIR / f"{part}.zip"
    print(f"downloading {url} ...")
    download_to(url, tmp_zip_path)

    all_rows = []
    all_records = []
    with zipfile.ZipFile(tmp_zip_path) as zf:
        doc_zip_names = [n for n in zf.namelist() if n.lower().endswith(".zip")]
        total = len(doc_zip_names) if MAX_DOCS_PER_PART is None else min(len(doc_zip_names), MAX_DOCS_PER_PART)
        print(f"  {len(doc_zip_names)} nested per-document zips found; processing {total}")

        n_processed = 0
        for doc_zip_name in doc_zip_names:
            if MAX_DOCS_PER_PART and n_processed >= MAX_DOCS_PER_PART:
                break
            try:
                nested_bytes = zf.read(doc_zip_name)
                with zipfile.ZipFile(BytesIO(nested_bytes)) as nested_zf:
                    nested_names = nested_zf.namelist()
                    xml_names = [n for n in nested_names if n.lower().endswith(".xml")]
                    for xml_name in xml_names:
                        records = parse_spl_bytes(nested_zf.read(xml_name), rx_or_otc=rx_or_otc)
                        for rec in records:
                            local_paths = []
                            for image_ref in rec.image_refs:
                                matches = [n for n in nested_names if n.split("/")[-1] == image_ref]
                                if matches:
                                    out_path = SPL_IMAGE_DIR / f"{rec.ndc}_{Path(image_ref).name}"
                                    if not out_path.exists():
                                        out_path.write_bytes(nested_zf.read(matches[0]))
                                    local_paths.append(str(out_path))
                            all_records.append(rec)
                            for path in local_paths:
                                all_rows.append({
                                    "path": path, "label": rec.ndc, "side": "unknown",
                                    "domain": "reference_pool", "source": "dailymed",
                                    "category": rec.rx_or_otc,
                                })
            except Exception:
                pass  # one bad nested zip/XML shouldn't kill the whole part
            n_processed += 1
            if n_processed % 1000 == 0:
                print(f"  ...{n_processed}/{total} processed, {len(all_rows)} images resolved so far")

    append_manifest_rows(all_rows)
    merge_metadata(all_records)
    tmp_zip_path.unlink(missing_ok=True)
    print(f"  done: {n_processed} documents, {len(all_rows)} images resolved, "
          f"{len(all_records)} pill records (metadata merged even without an image)")
    return len(all_rows)

state = load_state()
for part in SPL_PARTS_TO_FETCH:
    if state.get(part) == "done":
        print(f"skipping {part} (already completed in a prior run)")
        continue
    if part not in spl_zip_urls:
        print(f"skipping {part} (URL not found on the resources page)")
        continue
    rx_or_otc = "RX" if "rx" in part else "OTC"
    try:
        process_part(part, spl_zip_urls[part], rx_or_otc)
        state[part] = "done"
    except Exception as e:
        print(f"FAILED on {part}: {e}")
        state[part] = f"failed: {e}"
    save_state(state)

print("\nAll requested parts processed (or skipped/failed as logged above).")


## 4. Summary

In [ ]:
if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH) as f:
        rows = list(_csv.DictReader(f))
    print(f"Total manifest rows: {len(rows)}")
    from collections import Counter
    print("By category:", dict(Counter(r["category"] for r in rows)))
    print("By part (source is always 'dailymed'; check acquisition_state.json for per-part status)")
else:
    print("No manifest written yet — check the per-part logs above for errors.")

print("\nState:", json.load(open(STATE_PATH)) if STATE_PATH.exists() else "no state file")
print("\nNext step: commit this notebook (Save & Run All), then in the training")
print("notebook, Add Input -> Notebook -> this notebook, to read its output.")
